# Tree of Thoughts for Automated Code Debugging
**CS 4782 Final Project — Cornell University**

This notebook runs the full experiment pipeline:
1. Install Ollama + pull Qwen2.5-7B (free, open-source **general-purpose** model)
2. Load **DebugBench** (Logic Error category) — real LeetCode bugs
3. Run ToT-BFS, ToT-DFS, MCTS, and baselines (IO, CoT, CoT-SC)
4. Display results table and comparison plots

**Key parameters matching Yao et al. NeurIPS 2023:**
- k = 5 branching factor (paper uses b=5)
- depth = 3 levels: Area → Hypothesis → Fix
- Evaluator = LLM (sure / likely / impossible, like the paper)
- Model = `qwen2.5:7b` (general-purpose, not code-specialized)

> **Runtime**: Set to GPU (T4) for faster inference. Runtime → Change runtime type → T4 GPU.

## Step 1: Install Dependencies

In [ ]:
!pip install -q openai datasets matplotlib numpy tqdm

## Step 2: Install Ollama and Pull Qwen2.5-7B
This takes ~3-5 minutes to download the model (~4.7 GB). Run once per session.

> We use **qwen2.5:7b** (general-purpose) rather than the coder variant so that the structured
> search in ToT has real work to do — matching the paper's use of GPT-4 on Game of 24.

In [ ]:
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess
import time

# Start Ollama server in background
proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(3)
print("Ollama server started.")

In [ ]:
# Pull the model — this is the slow step (~5 min on Colab)
# Using general-purpose qwen2.5:7b, NOT the code-specialized coder variant.
# This matches the paper's design: use a model that can't trivially solve the task alone.
!ollama pull qwen2.5:7b

In [ ]:
# Verify model is available
!ollama list

## Step 3: Clone Repo and Set Up Paths

In [ ]:
import os, subprocess, sys

# Always start from /content so the clone lands in the right place
os.chdir('/content')

repo_dir = '/content/Tree-of-Thoughts-Debugger'
if not os.path.exists(repo_dir):
    subprocess.run(
        ['git', 'clone', 'https://github.com/Datboy0127/Tree-of-Thoughts-Debugger.git'],
        cwd='/content', check=True
    )
    print('Repository cloned.')
else:
    result = subprocess.run(['git', 'pull'], cwd=repo_dir, capture_output=True, text=True)
    print('Repository already exists — pulled latest:', result.stdout.strip() or 'up to date')

sys.path.insert(0, f'{repo_dir}/code')
os.chdir(repo_dir)
print('Working directory:', os.getcwd())
print('Python path includes:', repo_dir + '/code')


## Step 4: Quick Sanity Check (Demo Mode)

In [ ]:
# Test that the LLM client can reach Ollama
from llm_client import LLMClient
llm = LLMClient()  # defaults to ollama + qwen2.5:7b
resp = llm.call('Say hello in one sentence.', max_tokens=50)
print('Model response:', resp['content'])
print('Tokens used:', resp['tokens'])


## Step 5: Load Dataset
Downloads **DebugBench** from HuggingFace — real LeetCode bugs submitted by programmers.
We use the **Logic Error** category (hardest category, requires multi-step reasoning).

DebugBench stats:
- 4,253 bugs across Python3, Java, C++ (we use Python3)
- Logic Error: bugs that produce wrong output due to incorrect algorithm logic
- Each problem: buggy code + correct reference code + LeetCode problem description

In [ ]:
from data_loader import load_debugbench

NUM_PROBLEMS = 50  # increase to 164 for full run

problems = load_debugbench(n=NUM_PROBLEMS, bug_types=["Logic Error"], seed=42)
print(f"Loaded {len(problems)} problems")

# Bug type distribution
from collections import Counter
bug_types = Counter(p.bug_type for p in problems)
for bt, count in bug_types.most_common():
    print(f"  {bt:25s}: {count}")

In [ ]:
# Preview one problem
p = problems[0]
print(f"Task ID  : {p.task_id}")
print(f"Bug type : {p.bug_type}")
print(f"\n--- Buggy code ---\n{p.buggy_code}")
print(f"\n--- Test code (first 400 chars) ---\n{p.test_code[:400]}")

## Step 6: Run All Experiments
Runs ToT-BFS, ToT-DFS, MCTS, IO, CoT, and CoT-SC on all problems.

**Parameters (matching Yao et al., NeurIPS 2023):**
- k = 5 branching factor (paper's b=5)
- depth = 3 (Area → Hypothesis → Fix)
- Evaluator = LLM (sure/likely/impossible)
- MCTS: 12 simulations per problem

**Estimated time on Colab T4**: ~90-120 minutes total.

In [ ]:
from llm_client import LLMClient
from executor import CodeExecutor
from tot_debugger import ToTDebugger
from baselines import IOBaseline, CoTBaseline, CoTSCBaseline
from evaluate import save_results, compute_metrics
from tqdm.notebook import tqdm
import os

llm = LLMClient()
executor = CodeExecutor()
os.makedirs('results', exist_ok=True)

K = 5           # branching factor — matches paper's b=5
EVALUATOR = 'llm'  # sure/likely/impossible — matches paper exactly

def run_method(solver, problems, label):
    results = []
    for p in tqdm(problems, desc=label):
        try:
            r = solver.solve(p)
        except Exception as e:
            print(f"  ERROR {p.task_id}: {e}")
            from tot_debugger import DebugResult
            r = DebugResult(p.task_id, label, False, None, 0, 0, 0, 0.0, False, p.bug_type)
        results.append(r)
    m = compute_metrics(results)
    print(f"{label}: fix_rate={m['fix_rate']:.1%}  avg_tokens={m['avg_tokens']:.0f}")
    return results

all_results = {}

In [ ]:
# ToT - BFS (k=5, depth=3, LLM evaluator)
# Generates 5 area candidates → prunes impossible → 5 hypotheses per area → 5 fixes each
solver = ToTDebugger(llm, executor, k=K, search='bfs', evaluator=EVALUATOR)
all_results['tot-bfs-k5'] = run_method(solver, problems, 'tot-bfs-k5')
save_results(all_results['tot-bfs-k5'], 'results/tot-bfs-k5.json')

In [ ]:
# ToT - DFS (k=5, depth=3, LLM evaluator, with backtracking)
# Greedy selection at each level; backtracks at hypothesis level then area level
solver = ToTDebugger(llm, executor, k=K, search='dfs', evaluator=EVALUATOR)
all_results['tot-dfs-k5'] = run_method(solver, problems, 'tot-dfs-k5')
save_results(all_results['tot-dfs-k5'], 'results/tot-dfs-k5.json')

# ToT - MCTS (our novel contribution, 12 simulations)
# UCB1 selection: exploit winning hypotheses + explore untested ones
# Execution-based reward (pass/fail) backpropagated through tree
solver = ToTDebugger(llm, executor, k=K, search='mcts', n_simulations=12)
all_results['mcts-k5-s12'] = run_method(solver, problems, 'mcts-k5-s12')
save_results(all_results['mcts-k5-s12'], 'results/mcts-k5-s12.json')

In [ ]:
# Baseline: IO
all_results['io'] = run_method(IOBaseline(llm, executor), problems, 'io')
save_results(all_results['io'], 'results/io.json')

In [ ]:
# Baseline: CoT
all_results['cot'] = run_method(CoTBaseline(llm, executor), problems, 'cot')
save_results(all_results['cot'], 'results/cot.json')

In [ ]:
# Baseline: CoT-SC (5 samples)
all_results['cot-sc-5'] = run_method(CoTSCBaseline(llm, executor, n_samples=5), problems, 'cot-sc-5')
save_results(all_results['cot-sc-5'], 'results/cot-sc-5.json')

## Step 7: Results Table

In [ ]:
import json
from evaluate import compare_methods, print_comparison_table, compute_by_bug_type

comparison = compare_methods(all_results)
print_comparison_table(comparison)

# Save summary
with open('results/summary.json', 'w') as f:
    json.dump(comparison, f, indent=2)
print("\nSummary saved to results/summary.json")

In [ ]:
# Bug-type breakdown for ToT-BFS
print("\n── Bug-type breakdown (tot-bfs-k5) ──")
by_type = compute_by_bug_type(all_results['tot-bfs-k5'])
for bt, m in sorted(by_type.items()):
    bar = '█' * int(m['fix_rate'] * 20)
    print(f"  {bt:25s} {bar:<20s} {m['fix_rate']:.1%}  (n={m['n']})")

## Step 8: Visualizations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

methods = list(comparison.keys())
fix_rates   = [comparison[m]['fix_rate'] * 100 for m in methods]
first_rates = [comparison[m].get('first_attempt_rate', 0) * 100 for m in methods]

x = np.arange(len(methods))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w/2, fix_rates,   w, label='Fix Rate (%)',          color='#2196F3', alpha=0.9)
b2 = ax.bar(x + w/2, first_rates, w, label='1st-Attempt Rate (%)',  color='#FF9800', alpha=0.9)

ax.set_ylabel('Success Rate (%)', fontsize=13)
ax.set_title('Fix Rate by Method — DebugBench Logic Error (Qwen2.5-7B)', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(methods, rotation=15, ha='right', fontsize=11)
ax.set_ylim(0, 110)
ax.legend(fontsize=11)
ax.bar_label(b1, fmt='%.1f%%', padding=3, fontsize=9)
ax.bar_label(b2, fmt='%.1f%%', padding=3, fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
fig.tight_layout()
plt.savefig('results/fix_rates.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → results/fix_rates.png")

In [ ]:
# Token cost plot
tokens = [comparison[m]['avg_tokens'] for m in methods]
# Color: gray = baseline, green = ToT/MCTS
baseline_keys = {'io', 'cot', 'cot-sc-5'}
colors = ['#9E9E9E' if m in baseline_keys else '#4CAF50' for m in methods]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(methods, tokens, color=colors, alpha=0.85)
ax.set_ylabel('Avg Tokens per Problem', fontsize=13)
ax.set_title('Token Cost by Method', fontsize=14)
ax.set_xticklabels(methods, rotation=15, ha='right', fontsize=11)
ax.bar_label(bars, fmt='%.0f', padding=3, fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

baseline_patch = mpatches.Patch(color='#9E9E9E', label='Baseline')
tot_patch = mpatches.Patch(color='#4CAF50', label='ToT / MCTS')
ax.legend(handles=[baseline_patch, tot_patch], fontsize=11)
fig.tight_layout()
plt.savefig('results/token_cost.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → results/token_cost.png")

In [ ]:
# Bug-type breakdown (horizontal bar)
by_type = compute_by_bug_type(all_results['tot-bfs-k5'])
bug_labels = list(by_type.keys())
bug_rates  = [by_type[bt]['fix_rate'] * 100 for bt in bug_labels]
bug_ns     = [by_type[bt]['n'] for bt in bug_labels]

palette = ['#4CAF50', '#2196F3', '#FF9800', '#F44336', '#9C27B0']

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(bug_labels, bug_rates, color=palette[:len(bug_labels)], alpha=0.85)
ax.set_xlabel('Fix Rate (%)', fontsize=13)
ax.set_title('ToT-BFS (k=5) Fix Rate by Bug Type — DebugBench', fontsize=14)
ax.set_xlim(0, 110)
for i, (v, n) in enumerate(zip(bug_rates, bug_ns)):
    ax.text(v + 1.5, i, f'{v:.1f}%  (n={n})', va='center', fontsize=10)
ax.grid(axis='x', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
fig.tight_layout()
plt.savefig('results/bug_type_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → results/bug_type_breakdown.png")

In [ ]:
# Backtrack rate (DFS only)
dfs_results = all_results['tot-dfs-k5']
backtrack_counts = [r.backtracks for r in dfs_results]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Histogram of backtracks
axes[0].hist(backtrack_counts, bins=range(0, max(backtrack_counts)+2),
             color='#9C27B0', alpha=0.8, edgecolor='white')
axes[0].set_xlabel('Backtracks per Problem', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('DFS Backtrack Distribution', fontsize=13)
axes[0].grid(axis='y', alpha=0.3)

# Fix rate: problems that needed backtracking vs didn't
needed_bt  = [r for r in dfs_results if r.backtracks > 0]
no_bt      = [r for r in dfs_results if r.backtracks == 0]
labels = ['No backtrack', 'Backtracked']
rates  = [
    sum(r.success for r in no_bt)  / max(len(no_bt), 1) * 100,
    sum(r.success for r in needed_bt) / max(len(needed_bt), 1) * 100,
]
axes[1].bar(labels, rates, color=['#2196F3', '#FF5722'], alpha=0.85)
axes[1].set_ylabel('Fix Rate (%)', fontsize=12)
axes[1].set_title('DFS Fix Rate: Backtrack vs No Backtrack', fontsize=13)
axes[1].set_ylim(0, 110)
for i, v in enumerate(rates):
    axes[1].text(i, v + 2, f'{v:.1f}%', ha='center', fontsize=11)
axes[1].grid(axis='y', alpha=0.3)

fig.tight_layout()
plt.savefig('results/backtrack_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → results/backtrack_analysis.png")

## Step 9: MCTS Tree Visualization
Visualizes the search tree built by MCTS on a single example problem. Node size = visit count. Node color = win rate (green = high, red = low). Dashed border = the fix that passed all tests.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import numpy as np

def visualize_mcts_tree(solver, problem, n_simulations=12, k=5):
    """Run MCTS on one problem and draw the resulting search tree."""
    from tot_debugger import ToTDebugger

    # Run MCTS and capture the tree
    mcts = ToTDebugger(solver.llm, solver.executor, k=k, search='mcts', n_simulations=n_simulations)
    result = mcts.solve(problem)
    root = mcts._last_mcts_root

    children = root.children
    if not children:
        print("No tree to visualize.")
        return

    n = len(children)
    fig, ax = plt.subplots(figsize=(max(14, n * 2.8), 7))
    ax.set_xlim(-1, n)
    ax.set_ylim(-0.5, 2.2)
    ax.axis('off')
    fig.patch.set_facecolor('#F8F9FA')
    ax.set_facecolor('#F8F9FA')

    # ── Draw root node ────────────────────────────────────────────────────────
    root_x, root_y = (n - 1) / 2, 1.8
    root_circle = plt.Circle((root_x, root_y), 0.18, color='#455A64', zorder=3)
    ax.add_patch(root_circle)
    ax.text(root_x, root_y, 'ROOT\n(buggy code)', ha='center', va='center',
            fontsize=7.5, color='white', fontweight='bold', zorder=4,
            multialignment='center')

    # ── Color scale: red (0 wins) → green (all wins) ──────────────────────────
    def win_color(win_rate):
        r = int(220 * (1 - win_rate) + 30 * win_rate)
        g = int(80  * (1 - win_rate) + 180 * win_rate)
        b = 80
        return f'#{r:02x}{g:02x}{b:02x}'

    max_visits = max((c.visits for c in children), default=1)

    for i, node in enumerate(children):
        x = i
        y = 0.7

        win_rate = node.wins / node.visits if node.visits > 0 else 0
        color    = win_color(win_rate)

        # Node size scales with visits
        radius = 0.12 + 0.18 * (node.visits / max(max_visits, 1))

        # ── Edge from root ────────────────────────────────────────────────
        ax.annotate("", xy=(x, y + radius), xytext=(root_x, root_y - 0.18),
                    arrowprops=dict(arrowstyle='->', color='#90A4AE',
                                   lw=1.5 + node.visits * 0.3,
                                   connectionstyle='arc3,rad=0.0'))

        # ── Hypothesis node ───────────────────────────────────────────────
        circle = plt.Circle((x, y), radius, color=color, zorder=3, alpha=0.92)
        ax.add_patch(circle)

        # Dashed border if this node's fix passed
        if node.fix_code is not None and result.fix_code == node.fix_code:
            border = plt.Circle((x, y), radius + 0.025, fill=False,
                                linestyle='--', edgecolor='gold', linewidth=2.5, zorder=4)
            ax.add_patch(border)
            ax.text(x, y - radius - 0.12, '✓ SOLUTION', ha='center',
                    fontsize=8, color='#2E7D32', fontweight='bold')

        # Stats inside node
        stats = f"{node.wins:.0f}W/{node.visits}V\n{win_rate:.0%}"
        ax.text(x, y, stats, ha='center', va='center',
                fontsize=7, color='white', fontweight='bold', zorder=5,
                multialignment='center')

        # ── Hypothesis label below node ───────────────────────────────────
        # Wrap hypothesis text to ~35 chars
        hyp = node.hypothesis
        # Extract just the title portion (before EXPLANATION)
        title = hyp.split('EXPLANATION')[0].strip().replace('\n', ' ')
        if len(title) > 40:
            title = title[:38] + '…'

        ax.text(x, y - radius - 0.05, title, ha='center', va='top',
                fontsize=7.5, color='#212121', wrap=True,
                multialignment='center',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                          edgecolor=color, alpha=0.85))

        # ── UCB1 score label on edge ──────────────────────────────────────
        ucb = node.ucb1()
        ucb_str = f"UCB1={'∞' if ucb == float('inf') else f'{ucb:.2f}'}"
        mid_x = (root_x + x) / 2
        mid_y = (root_y + y) / 2
        ax.text(mid_x, mid_y + 0.05, ucb_str, ha='center', fontsize=6.5,
                color='#546E7A', style='italic',
                bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, pad=1))

    # ── Legend ────────────────────────────────────────────────────────────────
    low_patch  = mpatches.Patch(color=win_color(0.0), label='Win rate: 0%')
    mid_patch  = mpatches.Patch(color=win_color(0.5), label='Win rate: 50%')
    high_patch = mpatches.Patch(color=win_color(1.0), label='Win rate: 100%')
    gold_line  = mpatches.Patch(facecolor='white', edgecolor='gold',
                                linestyle='--', linewidth=2, label='Passing fix found')
    ax.legend(handles=[low_patch, mid_patch, high_patch, gold_line],
              loc='upper right', fontsize=8, framealpha=0.9)

    ax.set_title(
        f"MCTS Search Tree — {problem.task_id}  "
        f"({'✓ Solved' if result.success else '✗ Not solved'})  |  "
        f"{n_simulations} simulations, k={k}  |  "
        f"Node size ∝ visits",
        fontsize=12, pad=12, color='#212121'
    )

    plt.tight_layout()
    plt.savefig('results/mcts_tree.png', dpi=150, bbox_inches='tight', facecolor='#F8F9FA')
    plt.show()
    print(f"Saved → results/mcts_tree.png")
    print(f"Result: {'SOLVED' if result.success else 'not solved'} | "
          f"nodes explored={result.nodes_explored} | tokens={result.total_tokens}")


# Pick a problem and visualize
example_problem = problems[0]
print(f"Visualizing MCTS on: {example_problem.task_id} (bug type: {example_problem.bug_type})")
print(f"Buggy code:\n{example_problem.buggy_code}\n")

# Use the already-instantiated llm and executor
from tot_debugger import ToTDebugger as _ToT
_dummy_solver = _ToT(llm, executor, k=5)
visualize_mcts_tree(_dummy_solver, example_problem, n_simulations=12, k=5)

## Step 10: Save All Results and Download
Download the `results/` folder to use in your report and poster.

In [ ]:
import shutil

# Zip results folder for easy download
shutil.make_archive('/content/tot_results', 'zip', '/content/Tree-of-Thoughts-Debugger/results')
print("Results zipped → /content/tot_results.zip")

# Download (Colab only)
try:
    from google.colab import files
    files.download('/content/tot_results.zip')
except ImportError:
    print("Not running on Colab — find the zip at /content/tot_results.zip")

In [ ]:
# Final summary printout
print("=" * 55)
print(f"{'Method':<18} {'Fix Rate':>10} {'Avg Tokens':>12}")
print("=" * 55)
for method, m in comparison.items():
    print(f"{method:<18} {m['fix_rate']:>9.1%} {m['avg_tokens']:>12.0f}")
print("=" * 55)